## Assembling a unified database, beginning with E. Coli datasets
File structure:

In [ ]:
"""
This directory printout does not include filenames
used command: tree -d -I venv
-I ignores directories, -d prints only directories

└── data
    └── raw
        └── ecoli
            ├── cell_2018
            │   ├── bed
            │   └── bigwig
            ├── elife_2017
            │   ├── bed
            │   └── bigwig
            ├── mol_cell_2021
            │   ├── bed
            │   └── bigwig
            ├── nar_2021
            │   ├── bed
            │   └── bigwig
            ├── nature_struct_2016
            │   ├── bed
            │   └── bigwig
            ├── rna_biol_2022
            │   ├── bed
            │   └── bigwig
            └── science_2016
                ├── bed
                └── bigwig
"""


'\nThis directory printout does not print filenames\nused command: tree -d -I venv\n-I ignores directories, -d prints only directories\n\n└── data\n    └── raw\n        └── ecoli\n            ├── cell_2018\n            │   ├── bed\n            │   └── bigwig\n            ├── elife_2017\n            │   ├── bed\n            │   └── bigwig\n            ├── mol_cell_2021\n            │   ├── bed\n            │   └── bigwig\n            ├── nar_2021\n            │   ├── bed\n            │   └── bigwig\n            ├── nature_struct_2016\n            │   ├── bed\n            │   └── bigwig\n            ├── rna_biol_2022\n            │   ├── bed\n            │   └── bigwig\n            └── science_2016\n                ├── bed\n                └── bigwig\n'

In [2]:
import pandas as pd
import pyBigWig
import numpy as np
import os
from Bio import SeqIO

In [ ]:


root_dir = "/home/zofia/Research/datasetcreation/data/raw/ecoli"

unorganised_ecoli = {} #dict to store all the e coli files

# Loop through all folders and files
for dirpath, dirnames, filenames in os.walk(root_dir):
    for filename in filenames:
        file_path = os.path.join(dirpath, filename)
        # Only process bw files
        if filename.endswith(".bw"):
            #print(filename)
            bw = pyBigWig.open(f"{file_path}")
            #all chroms == U00096.2 which is the E. Coli reference chromosome
            intervals = bw.intervals('U00096.2')   # returns list of (start, end, value) or None
            unorganised_ecoli[filename] = intervals

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x712c60cbead0>>
Traceback (most recent call last):
  File "/home/zofia/Research/datasetcreation/venv/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


In [ ]:
#Exporting the data as it currently is. Consider not using a dataframe as runtime ~27s
ecoli_df = pd.DataFrame({key:pd.Series(value) for key, value in unorganised_ecoli.items()})
ecoli_df.to_csv('combined_datasets/raw_ecoli.csv', index=False)

In [5]:
ecoli_df = pd.read_csv("combined_datasets/raw_ecoli.csv")

print(ecoli_df.head())

ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

## Notes

Current raw database structure
- need to unwrap strings around each tuple of intervals
- should unify scores for each chromosome position but need to do that after normalising

File structure updates
- .json file stored in the top hierarchy of each study to read in as a dictionary containing:

In [2]:
{
    "DOI":"",
    "Tech":"",
    "Reagent":"",
    "Env":"", #in vivo, ex vivo, in vitro
    "Temp":"",

}

{'DOI': '', 'Tech': '', 'Reagent': '', 'Env': '', 'Temp': ''}

Pulling these out of the filenames with a regex would be straightforward rather than doing it manually.


### Notes on techniques & how to normalise/binarise them
SHAPE modifies the 2′-hydroxyl groups of ribose, while DMS targets adenines/cytosines, causing reverse transcription to stop at modified sites.
Issues: how to deal with overly reactive bases and drop outliers?

Potential tool: RNA Framework: an all in one toolkit for the analysis of RNA structures and post-transcriptional modification

Link: https://academic.oup.com/nar/article/46/16/e97/5035169?login=false 

The rf-norm module processes RNA structure probing experiments' RC files to perform reactivity normalization. To ensure the maximal analysis flexibility, rf-norm currently implements four different scoring and three different normalization schemes

Some notes on using BigWig files:


In [ ]:
header = bw.header()
print(header) # sumData and sumSquared are here so that we can compute statistics about the file
print(bw.chroms()) #chromosomes in the bigwig
chromosome = next(iter(bw.chroms())) #There is only one (circular) chromosome so I pull it out using an iterator
print(chromosome) #after checking the datasets, all store under bw.intervals('U00096.2') so can hard code this
print(len(bw.intervals(chromosome))) #the layout is always start, end, value (value of structure score)


{'version': 4, 'nLevels': 10, 'nBasesCovered': 35933, 'minVal': 0, 'maxVal': 51, 'sumData': 49319, 'sumSquared': 126703}
{'U00096.2': 4639675}
U00096.2
35933


In [1]:

"""
for chrom, length in bw.chroms().items():
    ints = bw.intervals(chrom)   # returns list of (start, end, value) or None
    if not ints:
        continue
    for s, e, v in ints:
        if v != v:   # faster NaN check: NaN != NaN
            continue
        print(chrom, s, e, v)


        #Use this to read into an array?"""

'\nfor chrom, length in bw.chroms().items():\n    ints = bw.intervals(chrom)   # returns list of (start, end, value) or None\n    if not ints:\n        continue\n    for s, e, v in ints:\n        if v != v:   # faster NaN check: NaN != NaN\n            continue\n        print(chrom, s, e, v)\n\n\n        #Use this to read into an array?'

Returning to this in november - adding other species of bacteria

In [ ]:
""".
├── combined_datasets
└── data
    └── raw
        ├── b.cereus
        │   └── nature_struct_2016
        ├── b.subtilis
        │   ├── nucleic_acids_2022
        │   └── rna_2021
        ├── ecoli
        │   ├── cell_2018
        │   │   ├── bed
        │   │   └── bigwig
        │   ├── elife_2017
        │   │   ├── bed
        │   │   └── bigwig
        │   ├── mol_cell_2021
        │   │   ├── bed
        │   │   └── bigwig
        │   ├── nar_2021
        │   │   ├── bed
        │   │   └── bigwig
        │   ├── nature_struct_2016
        │   │   ├── bed
        │   │   └── bigwig 
        │   ├── rna_biol_2022
        │   │   ├── bed
        │   │   └── bigwig
        │   └── science_2016
        │       ├── bed
        │       └── bigwig
        ├── p.putida
        │   └── science_2016
        ├── s.enterica
        │   └── biochem_2017
        ├── synechococcus
        │   └── science_2016
        └── y.pseudotuberculosis
            └── nar_2020

39 directories"""

b. cereus

In [3]:
root_dir = "/home/zofia/Research/datasetcreation/data/raw/b.cereus"

unorganised_b_cereus = {} #dict to store all the b.cereus files

# Loop through all folders and files
for dirpath, dirnames, filenames in os.walk(root_dir):
    for filename in filenames:
        file_path = os.path.join(dirpath, filename)
        # Only process bw files
        if filename.endswith(".bw"):
            #print(filename)
            bw = pyBigWig.open(f"{file_path}")
            #all chroms == AE017194.1 which is the particular strain of b.cereus - check paper
            intervals = bw.intervals('AE017194.1')   # returns list of (start, end, value) or None
            unorganised_b_cereus[filename] = intervals

In [4]:
b_cereus_df = pd.DataFrame({key:pd.Series(value) for key, value in unorganised_b_cereus.items()})
b_cereus_df.to_csv('combined_datasets/raw_b_cereus.csv', index=False)

b. subtilis

In [5]:
root_dir = "/home/zofia/Research/datasetcreation/data/raw/b.subtilis"

unorganised_b_subtilis = {} #dict to store all the b.subtilis files

# Loop through all folders and files
for dirpath, dirnames, filenames in os.walk(root_dir):
    for filename in filenames:
        file_path = os.path.join(dirpath, filename)
        # Only process bw files
        if filename.endswith(".bw"):
            #print(filename)
            bw = pyBigWig.open(f"{file_path}")
            #all chroms == NC_000964.3 which is the particular strain - check paper
            intervals = bw.intervals('NC_000964.3')   # returns list of (start, end, value) or None
            unorganised_b_subtilis[filename] = intervals

In [6]:
b_subtilis_df = pd.DataFrame({key:pd.Series(value) for key, value in unorganised_b_subtilis.items()})
b_subtilis_df.to_csv('combined_datasets/raw_b_subtilis.csv', index=False)

p.putida

In [3]:
root_dir = "/home/zofia/Research/datasetcreation/data/raw/p.putida"

unorganised_p_putida= {} #dict to store all the p.putida files

# Loop through all folders and files
for dirpath, dirnames, filenames in os.walk(root_dir):
    for filename in filenames:
        file_path = os.path.join(dirpath, filename)
        # Only process bw files
        if filename.endswith(".bw"):
            #print(filename)
            bw = pyBigWig.open(f"{file_path}")
            #all chroms == NC_002947.4 which is the particular strain - check paper
            intervals = bw.intervals('NC_002947.4')   # returns list of (start, end, value) or None
            unorganised_p_putida[filename] = intervals


p_putida_df = pd.DataFrame({key:pd.Series(value) for key, value in unorganised_p_putida.items()})
p_putida_df.to_csv('combined_datasets/raw_p_putida.csv', index=False)

s.enterica

In [4]:
root_dir = "/home/zofia/Research/datasetcreation/data/raw/s.enterica"

unorganised_s_enterica = {} #dict to store all the s.enterica files

# Loop through all folders and files
for dirpath, dirnames, filenames in os.walk(root_dir):
    for filename in filenames:
        file_path = os.path.join(dirpath, filename)
        # Only process bw files
        if filename.endswith(".bw"):
            #print(filename)
            bw = pyBigWig.open(f"{file_path}")
            #all chroms == NC_003197.2 which is the particular strain - check paper
            intervals = bw.intervals('NC_003197.2')   # returns list of (start, end, value) or None
            unorganised_s_enterica[filename] = intervals


s_enterica_df = pd.DataFrame({key:pd.Series(value) for key, value in unorganised_s_enterica.items()})
s_enterica_df.to_csv('combined_datasets/raw_s_enterica.csv', index=False)

synechococcus

In [5]:
root_dir = "/home/zofia/Research/datasetcreation/data/raw/synechococcus"

unorganised_synechococcus = {} #dict to store all the b.subtilis files

# Loop through all folders and files
for dirpath, dirnames, filenames in os.walk(root_dir):
    for filename in filenames:
        file_path = os.path.join(dirpath, filename)
        # Only process bw files
        if filename.endswith(".bw"):
            #print(filename)
            bw = pyBigWig.open(f"{file_path}")
            #all chroms == BX548020.1 which is the particular strain - check paper
            intervals = bw.intervals('BX548020.1')   # returns list of (start, end, value) or None
            unorganised_synechococcus[filename] = intervals


synechococcus_df = pd.DataFrame({key:pd.Series(value) for key, value in unorganised_synechococcus.items()})
synechococcus_df.to_csv('combined_datasets/raw_synechococcus.csv', index=False)

y.pseudotuberculosis

In [6]:
root_dir = "/home/zofia/Research/datasetcreation/data/raw/y.pseudotuberculosis"

unorganised_y_pseudotuberculosis = {} #dict to store all the y.pseudotuberculosis files

# Loop through all folders and files
for dirpath, dirnames, filenames in os.walk(root_dir):
    for filename in filenames:
        file_path = os.path.join(dirpath, filename)
        # Only process bw files
        if filename.endswith(".bw"):
            #print(filename)
            bw = pyBigWig.open(f"{file_path}")
            #all chroms == NC_010465 which is the particular strain - check paper
            intervals = bw.intervals('NC_010465')   # returns list of (start, end, value) or None
            unorganised_y_pseudotuberculosis[filename] = intervals


y_pseudotuberculosis_df = pd.DataFrame({key:pd.Series(value) for key, value in unorganised_y_pseudotuberculosis.items()})
y_pseudotuberculosis_df.to_csv('combined_datasets/raw_y_pseudotuberculosis.csv', index=False)

I'm wanting to check that all of the co-ordinate tuples are single nucleotides, and if so, unwrap them and keep only one co-ordinate for matching to the reference genome later. 

python validate_unwrap_coords.py \
  --in_dir combined_datasets \
  --pattern "raw_*.csv" \
  --output_suffix "_single_nt"

This script is not helpful because it uses the first col as canonical reference - and drops other co-ords - so a loci seen later would be lost. just using it to check data now. 